# Flagship Mock: LSST–DESI Catalog Visualization

Results from the `mock_lsst_desi_flagship` pipeline run.
Outputs are in `$LBG_DIR/shared/mock_lsst_desi_flagship/outputs/`.

| File | Description |
|---|---|
| `flagship_catalog.pq` | Noiseless Flagship reduced catalog (~252 M galaxies) |
| `lsst_y1_flagship.pq` | Flagship + LSST Y1 photometric errors |
| `lsst_y10_flagship.pq` | Flagship + LSST Y10 photometric errors |
| `lsst_y1_flagship_i25p3.pq` | Y1 mock, depth cut i < 25.3 |
| `lsst_y10_flagship_i26p5.pq` | Y10 mock, depth cut i < 26.5 |
| `desi_bgs_flagship.pq` | DESI BGS selection |
| `desi_lrg_flagship.pq` | DESI LRG selection |
| `desi_elg_flagship.pq` | DESI ELG selection |

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from astropy.wcs import WCS
from pathlib import Path

In [ ]:
LBG_DIR = %env LBG_DIR
LBG_DIR = Path(LBG_DIR)
BASE = LBG_DIR / "shared/mock_lsst_desi_flagship/outputs"

CATALOGS = {
    "flagship":      BASE / "flagship_catalog.pq",
    "lsst_y1":       BASE / "lsst_y1_flagship_i25p3.pq",
    "lsst_y10":      BASE / "lsst_y10_flagship_i26p5.pq",
    "bgs":           BASE / "desi_bgs_flagship.pq",
    "lrg":           BASE / "desi_lrg_flagship.pq",
    "elg":           BASE / "desi_elg_flagship.pq",
}

BANDS = ["u", "g", "r", "i", "z", "y"]
MAG_COLS = [f"mag_{b}_lsst" for b in BANDS]
ERR_COLS = [f"mag_{b}_lsst_err" for b in BANDS]

SAMPLE_SIZE = 1_000_000  # rows to draw from large catalogs for plotting


def sample_parquet(path, columns, n_target=SAMPLE_SIZE, seed=42):
    """Read ~n_target rows from a parquet file by sampling row groups.

    Parameters
    ----------
    path : str
        Path to the parquet file.
    columns : list of str
        Columns to read.
    n_target : int
        Approximate number of rows to return.
    seed : int
        Random seed for group selection.

    Returns
    -------
    pandas.DataFrame
    """
    pf = pq.ParquetFile(path)
    n_total = pf.metadata.num_rows
    n_groups = pf.metadata.num_row_groups
    if n_total <= n_target:
        return pf.read(columns=columns).to_pandas()
    rows_per_group = n_total / n_groups
    n_groups_needed = max(1, round(n_target / rows_per_group))
    rng = np.random.default_rng(seed)
    chosen = sorted(rng.choice(n_groups, size=min(n_groups_needed, n_groups), replace=False))
    tables = [pf.read_row_group(g, columns=columns) for g in chosen]
    return pa.concat_tables(tables).to_pandas()

## 1. Catalog Summary

Min/max redshift and LSST magnitude in each band for every catalog.
Row-group statistics are read directly from the Parquet metadata — no data is loaded.

In [ ]:
def parquet_minmax(path, cols):
    """Return {col: (min, max)} using per-row-group Parquet statistics.

    Parameters
    ----------
    path : str
        Path to the parquet file.
    cols : list of str
        Column names to summarise.

    Returns
    -------
    dict mapping column name to (min, max) tuple of floats
    """
    pf = pq.ParquetFile(path)
    meta = pf.metadata
    schema = pf.schema_arrow
    col_idx = {c: schema.get_field_index(c) for c in cols if schema.get_field_index(c) >= 0}
    result = {c: (np.inf, -np.inf) for c in col_idx}
    for rg in range(meta.num_row_groups):
        for c, idx in col_idx.items():
            stats = meta.row_group(rg).column(idx).statistics
            if stats is not None:
                lo, hi = result[c]
                result[c] = (min(lo, stats.min), max(hi, stats.max))
    return result


stat_cols = ["redshift"] + MAG_COLS

rows = []
for label, path in CATALOGS.items():
    mm = parquet_minmax(path, stat_cols)
    row = {"catalog": label}
    row["z_min"]  = round(mm["redshift"][0], 4)
    row["z_max"]  = round(mm["redshift"][1], 4)
    for b, col in zip(BANDS, MAG_COLS):
        row[f"{b}_min"] = round(mm[col][0], 2) if col in mm else np.nan
        row[f"{b}_max"] = round(mm[col][1], 2) if col in mm else np.nan
    rows.append(row)

summary = pd.DataFrame(rows).set_index("catalog")
summary

## 2. Flagship Spatial Density

The Flagship catalog covers six HEALPix pixels.
A naïve 2-D histogram in (RA, Dec) underestimates the density near the equator because equal-area pixels on the sphere map to unequal areas in RA–Dec.
We correct for this by dividing each bin by its solid angle $\Delta\Omega = \cos(\delta)\,\Delta\alpha\,\Delta\delta$, then display the result with a proper celestial projection (plate carrée / CAR) using `astropy.wcs` and `WCSAxes`.

In [ ]:
pos = sample_parquet(CATALOGS["flagship"], ["ra", "dec"])

ra  = pos["ra"].values
dec = pos["dec"].values

# Build a regular 2-D histogram in (RA, Dec)
nbin = 300
ra_edges  = np.linspace(ra.min(),  ra.max(),  nbin + 1)
dec_edges = np.linspace(dec.min(), dec.max(), nbin + 1)
H, _, _ = np.histogram2d(ra, dec, bins=[ra_edges, dec_edges])

# Correct each Dec strip for cos(dec) so the map shows surface density
dec_centers = 0.5 * (dec_edges[:-1] + dec_edges[1:])
cos_dec = np.cos(np.radians(dec_centers))  # shape (nbin,)
density = H / cos_dec[np.newaxis, :]  # broadcast over RA axis

# Field centre and pixel scale for the WCS
ra_cen  = 0.5 * (ra.min()  + ra.max())
dec_cen = 0.5 * (dec.min() + dec.max())
dpix_ra  = (ra.max()  - ra.min())  / nbin
dpix_dec = (dec.max() - dec.min()) / nbin

wcs = WCS(naxis=2)
wcs.wcs.crpix = [nbin / 2 + 1, nbin / 2 + 1]  # 1-indexed FITS convention
wcs.wcs.crval = [ra_cen, dec_cen]
wcs.wcs.cdelt = [-dpix_ra, dpix_dec]            # RA increases leftward
wcs.wcs.ctype = ["RA---CAR", "DEC--CAR"]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection=wcs)

# density.T has shape (nDec, nRA) as expected by imshow with origin='lower'
im = ax.imshow(
    density.T,
    origin="lower",
    cmap="magma",
    norm=mcolors.LogNorm(vmin=np.percentile(density[density > 0], 5)),
    interpolation="nearest",
)

ax.coords["ra"].set_axislabel("Right Ascension [deg]")
ax.coords["dec"].set_axislabel("Declination [deg]")
ax.coords["ra"].set_format_unit("deg")
ax.coords["dec"].set_format_unit("deg")
ax.grid(color="white", alpha=0.3, linestyle=":")
ax.set_title("Flagship catalog — galaxy surface density (1 M-galaxy sample)")
cb = fig.colorbar(im, ax=ax, pad=0.01)
cb.set_label(r"Relative surface density")
plt.tight_layout()
plt.show()

## 3. Flagship Catalog Distributions

Redshift distribution, per-band magnitude distributions, and color–redshift relations for the 1 M-galaxy sample drawn above.

In [ ]:
# Load flagship sample with all needed columns
fs_cols = ["redshift"] + MAG_COLS
fs = sample_parquet(CATALOGS["flagship"], fs_cols)

# --- Redshift distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fs["redshift"], bins=100, histtype="step", color="steelblue", linewidth=1.5)
ax.set_xlabel("Redshift")
ax.set_ylabel("Count")
ax.set_title("Flagship — redshift distribution (1 M-galaxy sample)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Per-band magnitude distributions ---
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
colors = ["#9B59B6", "#2ECC71", "#E74C3C", "#E67E22", "#1ABC9C", "#3498DB"]

for ax, band, col, color in zip(axes.flat, BANDS, MAG_COLS, colors):
    ax.hist(fs[col], bins=100, histtype="step", color=color, linewidth=1.5)
    ax.set_xlabel(f"mag_{band} [AB]")
    ax.set_ylabel("Count")
    ax.set_title(f"LSST {band}")

fig.suptitle("Flagship — LSST magnitude distributions (1 M-galaxy sample)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Color–redshift distributions ---
# Five adjacent-band colors vs redshift, displayed as 2-D density histograms.
adjacent_colors = [
    ("u-g", "mag_u_lsst", "mag_g_lsst"),
    ("g-r", "mag_g_lsst", "mag_r_lsst"),
    ("r-i", "mag_r_lsst", "mag_i_lsst"),
    ("i-z", "mag_i_lsst", "mag_z_lsst"),
    ("z-y", "mag_z_lsst", "mag_y_lsst"),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

z = fs["redshift"].values
for ax, (label, bluer, redder) in zip(axes, adjacent_colors):
    color = fs[bluer].values - fs[redder].values
    ax.hist2d(
        z, color,
        bins=150,
        cmap="inferno",
        norm=mcolors.LogNorm(),
        rasterized=True,
    )
    ax.set_xlabel("Redshift")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_ylim(-0.4, 3)

fig.suptitle("Flagship — color–redshift distributions (1 M-galaxy sample)", y=1.02)
plt.tight_layout()
plt.show()

## 4. LSST Photometric Error Distributions

Photometric noise was applied with `photerr.LsstErrorModel` at Y1 (`nYrObs=1`) and Y10 (`nYrObs=10`) depth.
Each panel shows the per-band 1-σ error as a function of observed (noisy) magnitude.
The depth-cut catalogs (i < 25.3 for Y1, i < 26.5 for Y10) are used here — they are the samples relevant for weak-lensing science.

In [ ]:
err_cols_needed = MAG_COLS + ERR_COLS

y1  = sample_parquet(CATALOGS["lsst_y1"],  err_cols_needed)
y10 = sample_parquet(CATALOGS["lsst_y10"], err_cols_needed)

fig, axes = plt.subplots(2, 6, figsize=(22, 7), sharex=True, sharey=True)
survey_data = [("Y1",  y1,  "royalblue"), ("Y10", y10, "tomato")]

for row_idx, (label, df, color) in enumerate(survey_data):
    for col_idx, (band, mag_col, err_col) in enumerate(zip(BANDS, MAG_COLS, ERR_COLS)):
        ax = axes[row_idx, col_idx]
        mag = df[mag_col].values
        err = df[err_col].values
        finite = np.isfinite(mag) & np.isfinite(err) & (err > 0)
        ax.scatter(
            mag[finite], err[finite],
            rasterized=True,
            s=1,
            color=color,
        )
        ax.set_xlabel(f"mag_{band} [AB]")
        if col_idx == 0:
            ax.set_ylabel(r"$\log_{10}(\sigma_\mathrm{mag})$")
        ax.set_title(f"LSST {band} — {label}")
        ax.set(xlim=(20, 30), ylim=(0, 1))

fig.suptitle("LSST photometric error vs. magnitude", y=1.01)
plt.tight_layout()
plt.show()

## 5. DESI Sample Redshift Distributions

BGS, LRG, and ELG galaxies are selected from the noiseless Flagship catalog using abundance-matched redshift-dependent halo-mass (BGS, LRG) or SFR (ELG) thresholds.
LRG is large (~98 M objects) so a 1 M-galaxy sample is used; BGS and ELG are small enough to load in full.

In [ ]:
bgs = sample_parquet(CATALOGS["bgs"], ["redshift"])  # tiny, loads fully
lrg = sample_parquet(CATALOGS["lrg"], ["redshift"])  # subsampled to 1 M
elg = sample_parquet(CATALOGS["elg"], ["redshift"])  # small, loads fully

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

desi_samples = [
    (axes[0], bgs, "BGS",  "seagreen",   (0.0, 0.6)),
    (axes[1], lrg, "LRG",  "firebrick",  (0.0, 1.2)),
    (axes[2], elg, "ELG",  "darkorchid", (0.0, 1.8)),
]

for ax, df, label, color, xlim in desi_samples:
    ax.hist(df["redshift"], bins=80, histtype="step", color=color, linewidth=1.5, density=True)
    ax.set_xlabel("Redshift")
    ax.set_ylabel("Normalized count")
    ax.set_title(f"DESI {label}")
    ax.set_xlim(xlim)

fig.suptitle("DESI sample redshift distributions", y=1.02)
plt.tight_layout()
plt.show()

## 6. BGS: Stellar Mass vs. r-band Magnitude

BGS targets bright, low-redshift galaxies.
`log_stellar_mass` is the log₁₀ stellar mass in solar masses from the Flagship simulation.

In [ ]:
bgs_full = sample_parquet(CATALOGS["bgs"], ["mag_r_lsst", "log_stellar_mass"])

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    bgs_full["mag_r_lsst"],
    bgs_full["log_stellar_mass"],
    c=bgs_full["mag_r_lsst"],
    cmap="plasma",
    s=2,
    alpha=0.5,
    linewidths=0,
    rasterized=True,
)
ax.set_xlabel(r"$r$-band magnitude [AB]")
ax.set_ylabel(r"$\log_{10}(M_\star / M_\odot)$")
ax.set_title("DESI BGS — stellar mass vs. r-band magnitude")
plt.colorbar(sc, ax=ax, label=r"$r$ [AB]")
plt.tight_layout()
plt.show()

## 7. LRG: r–z vs. z–y Color–Color

LRGs are red, massive galaxies.
The (r–z) vs. (z–y) diagram is used in DESI target selection to separate LRGs from stellar contamination.

In [ ]:
lrg_cols = ["mag_r_lsst", "mag_z_lsst", "mag_y_lsst", "redshift"]
lrg_s = sample_parquet(CATALOGS["lrg"], lrg_cols)

rz = lrg_s["mag_r_lsst"].values - lrg_s["mag_z_lsst"].values
zy = lrg_s["mag_z_lsst"].values - lrg_s["mag_y_lsst"].values

# Clip to the 1–99th percentile range in each color for display
rz_lim = np.percentile(rz, [1, 99])
zy_lim = np.percentile(zy, [1, 99])
mask = (
    (rz > rz_lim[0]) & (rz < rz_lim[1]) &
    (zy > zy_lim[0]) & (zy < zy_lim[1])
)

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(
    zy[mask], rz[mask],
    bins=200,
    cmap="inferno",
    norm=mcolors.LogNorm(),
    rasterized=True,
)
ax.set_xlabel(r"$z - y$")
ax.set_ylabel(r"$r - z$")
ax.set_title("DESI LRG — r–z vs. z–y (1 M-galaxy sample)")
fig.colorbar(h[3], ax=ax, label="Count")
plt.tight_layout()
plt.show()

## 8. ELG: g–r vs. r–z Color–Color

ELGs are star-forming galaxies with strong emission lines.
The (g–r) vs. (r–z) diagram separates ELGs from LRGs and quiescent galaxies; ELGs occupy a distinctive blue cloud.

In [ ]:
elg_cols = ["mag_g_lsst", "mag_r_lsst", "mag_z_lsst"]
elg_full = sample_parquet(CATALOGS["elg"], elg_cols)  # 396k rows, loads fully

gr = elg_full["mag_g_lsst"].values - elg_full["mag_r_lsst"].values
rz = elg_full["mag_r_lsst"].values - elg_full["mag_z_lsst"].values

gr_lim = np.percentile(gr, [1, 99])
rz_lim = np.percentile(rz, [1, 99])
mask = (
    (gr > gr_lim[0]) & (gr < gr_lim[1]) &
    (rz > rz_lim[0]) & (rz < rz_lim[1])
)

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(
    rz[mask], gr[mask],
    bins=100,
    cmap="viridis",
    norm=mcolors.LogNorm(),
    rasterized=True,
)
ax.set_xlabel(r"$r - z$")
ax.set_ylabel(r"$g - r$")
ax.set_title("DESI ELG — g–r vs. r–z")
fig.colorbar(h[3], ax=ax, label="Count")
plt.tight_layout()
plt.show()